# The MasakhaNER Benchmark, Run and Scored

**Computational Text Analysis Pack · Module 23 — African NLP Models and Benchmarks**

*Technique:* Token classification against gold labels; entity-level evaluation on a real benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/23c_masakhaner_benchmark.ipynb)

Data: `masakhaner.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhaner.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_ner   = pd.read_csv('masakhaner.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_ner has {len(df_ner):,} rows and {df_ner.shape[1]} columns.')

## What this notebook does

`masakhaner.csv` is not a dataset we made up for teaching. It is **MasakhaNER
2.0**, the benchmark African NLP models are scored on, built by the Masakhane
community: 147,053 hand-annotated sentences across 19 languages, with the
official train/dev/test split preserved.

This notebook runs a model that was fine-tuned on that benchmark and scores it
against the gold labels — the actual evaluation loop, on the actual benchmark.

## The tooling

**HuggingFace `transformers`** is the working layer for every model here: it
downloads the weights, tokenises, and runs the forward pass. Underneath it sits
**PyTorch** (the research default, dynamic and easy to debug); **TensorFlow /
Keras** is the other option, and what the Case Studies module uses.

The models themselves come from African NLP research groups:

- **AfriBERTa** (Ogueji et al.) — trained on 11 African languages, deliberately
  small, and evidence that useful models do not need web-scale data.
- **AfroXLMR** (Alabi et al.) — XLM-R adapted to African languages; the NER model
  in this notebook is its fine-tuned descendant.
- **LaBSE** (Google) — 109 languages, sentence-level, and the reason a Hausa
  sentence and its English translation land in the same place.
- **MasakhaNER / MasakhaNEWS / MasakhaPOS** — the benchmarks, built by the
  Masakhane community. `masakhaner.csv` in this course *is* that benchmark.

### Where the GPU comes from, and why you never need a key

The setup cell picks a backend from wherever you are running:

| Running in | GPU | What you need |
|---|---|---|
| **CourseHub Lab** | a shared L40S, reached through an authenticated CourseHub endpoint | nothing — the credential stays on the server |
| **Colab** | *your own* free GPU (`Runtime > Change runtime type > T4`) | nothing |
| **Anywhere offline** | none; cells use vectors shipped with the course | nothing |

There is no API key in this course, by design. A notebook cell is readable by
whoever runs it, so a key pasted into one is a key given away.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
BACKEND = "cached"
_pod = _hf = None

if IN_COLAB:
    # Colab hands every participant their own GPU. Use it rather than sharing
    # one server: it is faster for a room full of people, and needs no key.
    try:
        import transformers  # noqa: F401
    except ImportError:
        os.system("pip install -q transformers")
    import torch
    BACKEND = "colab-gpu" if torch.cuda.is_available() else "colab-cpu"
else:
    try:
        sys.path.insert(0, "/app")      # the CourseHub runner executes inside /app
        from utils import gpu_pod as _pod
        _info = _pod.health()
        BACKEND = "coursehub-pod"
    except Exception as exc:
        _why = str(exc)[:80]

POD_LIVE = BACKEND in ("coursehub-pod", "colab-gpu", "colab-cpu")

def _hf_pipe(kind, repo):
    global _hf
    _hf = _hf or {}
    if (kind, repo) not in _hf:
        from transformers import AutoModel, AutoTokenizer, pipeline
        import torch
        dev = 0 if torch.cuda.is_available() else -1
        if kind == "ner":
            _hf[(kind, repo)] = pipeline("token-classification", model=repo,
                                         aggregation_strategy="simple", device=dev)
        else:
            _hf[(kind, repo)] = (AutoTokenizer.from_pretrained(repo),
                                 AutoModel.from_pretrained(repo))
    return _hf[(kind, repo)]

def embed(texts, model="labse"):
    """Sentence embeddings, wherever we are. Returns a numpy array."""
    import numpy as np
    if BACKEND == "coursehub-pod":
        return np.asarray(_pod.embed(list(texts), model=model)["vectors"], dtype="float32")
    if BACKEND.startswith("colab"):
        import torch
        tok, mod = _hf_pipe("embed", "sentence-transformers/LaBSE")
        if torch.cuda.is_available():
            mod = mod.cuda()
        with torch.no_grad():
            b = tok(list(texts), padding=True, truncation=True,
                    max_length=128, return_tensors="pt")
            b = {k: v.to(mod.device) for k, v in b.items()}
            out = mod(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
            v = torch.nn.functional.normalize(v, dim=-1)
        return v.cpu().numpy().astype("float32")
    raise RuntimeError("no GPU backend available; use the shipped vectors instead")

def ner(texts, model="masakhaner"):
    """Named entities from the MasakhaNER 2.0 benchmark model."""
    repo = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
    if BACKEND == "coursehub-pod":
        return _pod.ner(list(texts), model=model)["results"]
    if BACKEND.startswith("colab"):
        pipe = _hf_pipe("ner", repo)
        out = pipe(list(texts))
        if out and isinstance(out[0], dict):
            out = [out]
        return [[{"text": e["word"], "label": e["entity_group"],
                  "score": round(float(e["score"]), 4)} for e in ents] for ents in out]
    raise RuntimeError("no GPU backend available")

print(f"backend: {BACKEND}")
if BACKEND == "coursehub-pod":
    print(f"   {_info['gpu']}, {_info['vram_total_mb']:,} MB VRAM (shared, via CourseHub)")
elif BACKEND == "colab-gpu":
    import torch; print(f"   {torch.cuda.get_device_name(0)} (your own Colab GPU)")
elif BACKEND == "colab-cpu":
    print("   Colab without a GPU — Runtime > Change runtime type > T4 GPU makes this faster")
else:
    print(f"   no live backend ({_why}); cells fall back to the vectors shipped with the course")

In [ ]:
print(f"{len(df_ner):,} sentences, {df_ner['language'].nunique()} languages, "
      f"{int(df_ner['n_entities'].sum()):,} entities")
print()
print(df_ner.groupby("language")
      .agg(sentences=("tokens", "size"), entities=("n_entities", "sum"))
      .sort_values("sentences", ascending=False).head(10).to_string())
print()
print("splits:", df_ner["split"].value_counts().to_dict())

## The BIO tagging scheme

Entities span several tokens, so each token gets a tag: `B-PER` begins a person,
`I-PER` continues it, `O` is outside any entity. This is the same sequence
labelling problem module 16 solved with an HMM — same shape, different machinery.

In [ ]:
row = df_ner[(df_ner["language"] == "Hausa") & (df_ner["n_entities"] >= 2)].iloc[0]
toks = row["tokens"].split()
tags = row["tags"].split()
print(f"{row['language']} ({row['split']} split)\n")
for t, g in zip(toks, tags):
    marker = "  <--" if g != "O" else ""
    print(f"  {t:<22}{g}{marker}")

def spans(tokens, tags):
    out, cur, lab = [], [], None
    for t, g in zip(tokens, tags):
        if g.startswith("B-"):
            if cur: out.append((" ".join(cur), lab))
            cur, lab = [t], g[2:]
        elif g.startswith("I-") and cur:
            cur.append(t)
        else:
            if cur: out.append((" ".join(cur), lab))
            cur, lab = [], None
    if cur: out.append((" ".join(cur), lab))
    return out

print("\ngold entities:", spans(toks, tags))

## Running the benchmark model

`masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0` is AfroXLMR fine-tuned on this
benchmark. We score it on held-out **test** sentences, entity-level: a
prediction counts only if the span *and* the label both match.

In [ ]:
# Every language MasakhaNER 2.0 covers that has enough held-out data.
# Trim this list if you are short of time; leave it to see the spread.
LANGS = sorted(df_ner[df_ner["split"] == "test"]["language"].unique())
PER_LANG = 8      # <- raise this for steadier numbers, lower it to go faster
print(f"{len(LANGS)} languages: {', '.join(LANGS)}")

test = df_ner[df_ner["split"] == "test"]
sample = (test[test["language"].isin(LANGS)]
          .groupby("language", group_keys=False).head(PER_LANG).reset_index(drop=True))
print(f"scoring {len(sample)} held-out sentences across {len(LANGS)} languages")

results = None
if POD_LIVE:
    results = []
    for i in range(0, len(sample), 32):
        chunk = [" ".join(t.split()) for t in sample["tokens"].iloc[i:i+32]]
        results.extend(ner(chunk))
    print("scored on the live GPU")
else:
    print("Pod offline — the scoring cell below is skipped, and the gold-label")
    print("analysis still runs so the lesson is not empty.")

In [ ]:
if results is not None:
    rows = []
    for i, row in sample.iterrows():
        gold = {(s.lower(), l) for s, l in spans(row["tokens"].split(), row["tags"].split())}
        pred = {(e["text"].strip().lower(), e["label"]) for e in results[i]}
        rows.append({"language": row["language"],
                     "gold": len(gold), "pred": len(pred),
                     "correct": len(gold & pred)})
    sc = pd.DataFrame(rows).groupby("language").sum(numeric_only=True)
    sc["precision"] = sc["correct"] / sc["pred"].clip(lower=1)
    sc["recall"] = sc["correct"] / sc["gold"].clip(lower=1)
    sc["F1"] = 2 * sc["precision"] * sc["recall"] / (sc["precision"] + sc["recall"]).clip(lower=1e-9)
    print("Entity-level scores on held-out MasakhaNER test sentences:\n")
    print(sc.to_string(float_format=lambda v: f"{v:.3f}"))
    print()
    print("Published MasakhaNER 2.0 F1 for strong models sits around 0.80-0.90")
    print("depending on language. Scores well below that here usually mean")
    print("tokenisation trouble, not a broken model — see the next cell.")
else:
    print("(skipped: pod offline)")

## The failure worth seeing

Run the cell below and read the Yorùbá output carefully. The model handles Hausa
and Swahili names cleanly and then **splits Yorùbá names across their
diacritics** — `Adéyẹmí` comes back as two fragments.

That is module 3's ASCII trap, alive inside a current African NLP model. The
tokeniser was not built with Yorùbá orthography in mind, and every downstream
number inherits the damage. It is the single most important thing in this
notebook for anyone planning to use these tools on Yorùbá material.

In [ ]:
# Real held-out sentences, one per language, so the comparison is across the
# actual benchmark rather than sentences chosen to make a point.
probe_rows = (df_ner[(df_ner["split"] == "test") & (df_ner["n_entities"] >= 2)]
              .groupby("language", group_keys=False).head(1))
if POD_LIVE:
    texts = [" ".join(t.split()) for t in probe_rows["tokens"]]
    out = ner(texts)
    print(f"{'language':<18}{'gold':>5}{'found':>7}   entities the model returned")
    print("-" * 96)
    for (_, r), ents in zip(probe_rows.iterrows(), out):
        got = ", ".join(f"{e['text']}[{e['label']}]" for e in ents[:5]) or "(none)"
        print(f"{r['language']:<18}{int(r['n_entities']):>5}{len(ents):>7}   {got[:60]}")
    print()
    print("Scan the middle columns. Where 'found' greatly exceeds 'gold', the")
    print("model is splitting single names into fragments — which happens most in")
    print("languages whose orthography uses diacritics or special characters.")
    print("Yoruba is the clearest case; compare it against Hausa and Swahili.")
else:
    print("Pod offline. Recorded result from this model:")
    print("  Hausa   -> 'Bola Tinubu'[PER], 'Kano'[LOC], 'ranar Litinin'[DATE]")
    print("  Swahili -> 'Tanzania'[LOC], 'Samia Suluhu Hassan'[PER], 'Dodoma'[LOC]")
    print("  Yoruba  -> 'A'[PER], 'deyemi'[PER], ''[LOC], 'Oyo'[LOC]  <-- fragmented")

### 🔧 Your turn 1

Add a language of your own to LANGS and score it. Does its F1 match the published benchmark figures?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually it is lower, for two reasons worth separating. First, we score on a
small sample, so the estimate is noisy — increase PER_LANG and it steadies.
Second, and more importantly, our span matching is strict and case-folded while
the official evaluation uses `seqeval` over BIO sequences, which treats
boundaries slightly differently. Reproducing a published benchmark number
exactly requires the published evaluation script, and "my F1 doesn't match the
paper" is far more often an evaluation-protocol difference than a model
difference.

</details>